In [1]:
# Libraries import

import requests
import os
import pygsheets
import pandas as pd
from dotenv import load_dotenv
from pangres import upsert
from sqlalchemy import text, create_engine

# Load environment variables
try:
    load_dotenv()
    print("Environment variables loaded successfully.")
except Exception as e:
    print("Failed to load environment variables:", e)
    raise

try:
    API_KEY = os.getenv('riot_api_key')
    username = os.getenv('db_username')
    password = os.getenv('db_password')
    host = os.getenv('db_host')
    port = os.getenv('db_port')
    name = os.getenv('db_name')

    # Check if any of the required variables are missing
    if not all([API_KEY, username, password, host, port, name]):
         raise ValueError("One or more required environment variables are missing.")

    print("Required environment variables are set.")
except Exception as e:
    print("Error retrieving environment variables:", e)
    raise

# Create a database connection string

def create_db_connection_string(username, password, host, port, name):
    try:
        connection_url = f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{name}"
        print("Database connection string created successfully.")
        return connection_url
    except Exception as e:
        print("Error creating DB connection string:", e)
        raise

try:
    conn = create_db_connection_string(username, password, host, port, name)
    db_engine = create_engine(conn, pool_recycle=3600)
    print("Database engine created successfully.")
except Exception as e:
    print("Error creating database engine:", e)
    raise

try:
    connection = db_engine.connect()
    print("Database connection established successfully.")
except Exception as e:
    print("Error connecting to the database:", e)
    raise

Environment variables loaded successfully.
Required environment variables are set.
Database connection string created successfully.
Database engine created successfully.
Database connection established successfully.


In [4]:
# Get Challengers summonerId with improved error handling

def get_ladder(top=None):
    # These constant values don't need error handling because they don't change.
    root = 'https://br1.api.riotgames.com/tft/'
    challenger = 'league/v1/challenger?queue=RANKED_TFT'
    grandmaster = 'league/v1/grandmaster?queue=RANKED_TFT'
    master = 'league/v1/master?queue=RANKED_TFT'
    
    try:
        # Get challenger data
        challenger_response = requests.get(root + challenger + '&api_key=' + API_KEY)
        challenger_response.raise_for_status()
        challenger_df = pd.DataFrame(challenger_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)
        
        grandmaster_df = pd.DataFrame()
        master_df = pd.DataFrame()
        
        # Get grandmaster data if applicable
        if top > 50:
            grandmaster_response = requests.get(root + grandmaster + '&api_key=' + API_KEY)
            grandmaster_response.raise_for_status()
            grandmaster_df = pd.DataFrame(grandmaster_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)
        
        # Get master data if applicable
        if top > 150:
            master_response = requests.get(root + master + '&api_key=' + API_KEY)
            master_response.raise_for_status()
            master_df = pd.DataFrame(master_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)

        ladder = pd.concat([challenger_df, grandmaster_df, master_df])[:top].reset_index(drop=True)
        ladder = ladder.reset_index(drop=False).drop(columns='rank').rename(columns={'index':'rank'})
        ladder['rank'] += 1

        return ladder

    except Exception as err:
        print("Error occurred while fetching ladder data:", err)
        return None

In [5]:
df_ladder = get_ladder(top=250)
df_ladder = df_ladder.set_index('rank')
df_ladder

,summonerId,puuid,leaguePoints,wins,losses,veteran,inactive,freshBlood,hotStreak
rank,,,,,,,,,
1,8ZNCE5oDTwbMwVBmhOZzhOxXJampyL-SWWvA1ZkidSFytQ,YnFAk-jtL0LKPGysuiXl7txhzZjg52AujkEoQT-19MoIi5...,1764,444,231,True,False,False,True
2,09xgBUvPluRyg4LSg9CgpgkBkZjWDc11EvEU_VI1DeU_mUs,QNCjsdXAx_gUj0oLNUnngBLdSqPxHhG4jodhzkBRUoPVp3...,1753,344,224,True,False,False,True
3,Ed8sP7f54PmEvg7lVplD6mCcRuybCXD_tCUt6obwImbJ,wcCl4tWDpoOOF8hbG5fo9GH7HIh8CLkDCMEY2Lq9h_ORqp...,1744,311,184,True,False,False,False
4,HHzxnU-gmYi4jX3TPyB_T-Ktp_y2n4ByVHUcrzDZTF4dx_0,zBZDYfnwAjB9oRgECZJYT-IdK64eoOnZJCgPAAiVZuF4k5...,1736,588,329,True,False,False,False
5,ZnxIMwoDhM-iowktvvR0vZy5vvEIF8ynTPD92DVwdlwtdA,NAyJ38QMUuM7BjeduvCFBWYal843yOm-gjYUJtvBn5sgsG...,1597,269,149,True,False,False,False
...,...,...,...,...,...,...,...,...,...
246,C4BKbJWA6Xwi0PmI18B53Vjx1t4YeLD1aHYwxLTbxfJhEQ,A2C-rVHsuMbqcDzEkXUS2pWwFTpnn9r0ZaOpkLI5z8ouua...,591,111,78,True,False,False,False
247,z0iDQW9U6lQbYOGVrHuzgK98_LtUf17N8-rFOyx1EYaxBw,PZBLWpfbhbNw4KENA6lNfI3_ES9tZhtPbF2dVNaq59Xvv1...,591,240,160,False,False,True,False
248,YmUlZV_y4Rc5vhH8aZ-lgSC5B9ahPn4RIGLm1wF4Zjm-,1Q8dmT6lsbxU9NzCFGNmYL6LjpgvZAcy-KL3Z8tE4HAsVS...,590,300,263,True,False,False,False


In [9]:
upsert(con=connection, df=df_ladder, schema='tft_ranked', table_name='ranked_ladder', create_table=True, create_schema=True, if_row_exists='update')

In [10]:
connection.commit()

In [7]:
with db_engine.connect() as conn:
    df_query = pd.read_sql(text("SELECT * FROM tft_ranked.ranked_ladder"), conn)

In [8]:
df_query

,rank,summonerId,leaguePoints,wins,losses,veteran,inactive,freshBlood,hotStreak
0,1,81TNb5AeP0llAXhrPPctqpn29ySkvIF9j_PJHLEOvMDXAeo,1789,224,97,True,False,False,False
1,2,h7ZPEa9uBc7YUhKgYYqSrVFK-Kt-lLQeDBKDP66NVROn-w,1732,158,57,False,False,False,True
2,3,8ZNCE5oDTwbMwVBmhOZzhOxXJampyL-SWWvA1ZkidSFytQ,1651,269,131,True,False,False,False
3,4,FCA6EjmiXnKVZd3YGVykmsSJCkS9MZ3a6ecZCWsso6JHtLY,1633,176,78,True,False,False,True
4,5,HHzxnU-gmYi4jX3TPyB_T-Ktp_y2n4ByVHUcrzDZTF4dx_0,1632,496,299,True,False,False,False
...,...,...,...,...,...,...,...,...,...
245,246,RC9PJx2_FBuPY16iqaMlOAbDo0iWQ-DyuyO94u7jgxevmA,340,160,141,True,False,False,False
246,247,1nQUl8yo3SQz-Qs-FhMRaspj9VpYdB3wZlIPWC3ZfhNKbQ,340,140,94,False,False,False,False
247,248,0dFnH9ZMoDPnhEAIUqWSimqtnB0yPypB6nTK1gY3uc_pwCo,338,198,188,True,False,False,False
248,249,YSvDnL_5R8HAyuehkplKkvjqHFZb0OgW8gGld1TDDnIvoOQ,336,93,48,False,False,False,True


In [ ]:
game_name = 'LustGuard'
tag_line = 'BR1'

def get_puuid(game_name=None, tag_line=None, API_KEY=None):
    link = f'https://americas.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{game_name}/{tag_line}?api_key={API_KEY}'
    response = requests.get(link)
    return response.json()['puuid']

In [ ]:
get_puuid(game_name=game_name, tag_line=tag_line, API_KEY=API_KEY)

In [ ]:
temp_df = get_ladder(top=100)['summonerId']

In [ ]:
puuid_dict = {}
def get_puuid(df):
    root = 'https://br1.api.riotgames.com/tft/league/v1/entries/by-summoner/'
    for summoner in temp_df:
        response = requests.get(root + summoner + '?api_key=' + API_KEY)
        if response.status_code == 200:
            data = response.json()
            summoner_puuid = data[0].get('puuid')
            puuid_dict[summoner] = summoner_puuid
        else:
            print(f'Error to retrieve the data from {summoner}: {response.status_code}')
    return puuid_dict

In [ ]:
df = get_puuid(temp_df)
df